<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# Duckietown and ROS 2

In this notebook we use what we have learned about ROS 2 in the previous notebooks to put everything together on the Duckiebot, either in the Duckiematrix or on real hardware: a tour of the Duckietown message package, the anatomy of the launch file that starts your node, and the complete `dts code build` / `dts code workbench` workflow that deploys it.

## The `duckietown_msgs` package

The message types of the Duckiebot are not part of ROS 2; they are defined by Duckietown in the package `duckietown_msgs` of the [`dt-ros2-commons`](https://github.com/duckietown/dt-ros2-commons/tree/ente/packages/duckietown_msgs/msg) repository, from which every Duckietown ROS 2 image (and this editor) is built. List them:

```bash
ros2 interface list | grep duckietown_msgs
```

About eighty definitions appear, grouped by what they describe. The ones this learning experience used, and a few you will meet in other learning experiences:

| Message | Published by | Meaning |
| --- | --- | --- |
| `Twist2DStamped` | your nodes, lane following, the joystick mapper | car command: `v` (m/s), `omega` (rad/s) |
| `WheelsCmdStamped` | `kinematics_node` | command for each wheel: `vel_left`, `vel_right` (-1 to 1) |
| `WheelEncoderStamped` | the encoder drivers | tick count and resolution of a wheel |
| `Pose2DStamped` | `velocity_to_pose_node` | planar pose `x`, `y`, `theta` integrated from the commands |
| `LEDPattern` | your nodes, to `led_driver_node/led_pattern` | colours and blinking frequency of the LEDs |
| `LanePose` | the lane filter | lateral offset `d` and heading `phi` in the lane |
| `FSMState` | the finite state machine | the current mode of the robot (`car_cmd_switch_node` listens to it) |

A message definition is a text file with one field per line, `type name`, optionally with constants (`uint8 ENCODER_TYPE_INCREMENTAL=1`) and nested messages (`std_msgs/Header header`):

```bash
ros2 interface show duckietown_msgs/msg/LEDPattern
ros2 interface show duckietown_msgs/msg/Pose2DStamped
```

Field types are the primitive types (`bool`, `int8`...`int64`, `uint8`...`uint64`, `float32`, `float64`, `string`), arrays of them (`string[] color_list`), and other messages. Services (`.srv`) and actions (`.action`) are defined in the same way, with sections separated by `---`:

```bash
ros2 interface show std_srvs/srv/Trigger
```

```shell
---
bool success
string message
```

An empty request, and a response with `success` and `message`: the service type used by `drive_node/start` and `stop`.

### Defining your own messages

Message definitions are compiled into Python and C++ code at build time, which pure-Python (`ament_python`) packages such as `ros2_basics` cannot do. A custom message lives in its own `ament_cmake` package, with the `.msg` files in a `msg/` directory, `rosidl_generate_interfaces(...)` in `CMakeLists.txt` and `<member_of_group>rosidl_interface_packages</member_of_group>` in `package.xml`; after a build, `ros2 interface show my_msgs/msg/MyMessage` prints it and `from my_msgs.msg import MyMessage` imports it. In Duckietown, new messages are added to `duckietown_msgs` so that every image shares them; the [ROS 2 tutorial on custom interfaces](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Custom-ROS2-Interfaces.html) walks through the steps.

## Anatomy of the launch file

Every node of this learning experience is started by a launch file, which is a Python script. [launch/drive_node.launch.py](../packages/ros2_basics/launch/drive_node.launch.py) is the one the recipe's launcher runs; read it line by line.

```python
from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument
from launch.substitutions import EnvironmentVariable, LaunchConfiguration
from launch_ros.actions import Node
```

`launch` is the generic launch system; `launch_ros` adds the ROS-specific `Node` action. A launch file must define `generate_launch_description()` returning a `LaunchDescription`, a list of actions.

```python
veh = LaunchConfiguration("veh")
car_cmd_topic = LaunchConfiguration("car_cmd_topic")
config = os.path.join(get_package_share_directory("ros2_basics"), "config", "drive_node", "default.yaml")
```

A `LaunchConfiguration` is the value of a launch argument, resolved when the launch runs. `get_package_share_directory` finds the installed `share/ros2_basics` directory, where `setup.py` installed the yaml file (notebook 05).

```python
DeclareLaunchArgument("veh", default_value=EnvironmentVariable("VEHICLE_NAME"), ...)
```

The `veh` argument can be given on the command line (`veh:=ROBOTNAME`) and defaults to the `VEHICLE_NAME` environment variable, which is set in every Duckietown container and in the editor terminals connected to a robot.

```python
Node(
    package="ros2_basics",
    executable="drive_node",
    name="drive_node",
    namespace=veh,
    parameters=[config],
    remappings=[("car_cmd_switch_node/cmd", car_cmd_topic)],
    output="screen",
    emulate_tty=True,
)
```

One node: the executable `drive_node` of the package `ros2_basics` (from `setup.py`'s `console_scripts`), named `drive_node`, in the namespace of the robot, with the parameters of the yaml file, the optional remapping of notebook 11, and its log printed to the terminal (`output="screen"`; `emulate_tty` keeps the colours). A launch file can start many nodes at once, include other launch files and set parameters inline (`parameters=[config, {"v": 0.3}]`); the robot's own stacks are started this way.

The yaml file applies to the node through `parameters=[config]`:

```yaml
/**:
  ros__parameters:
    v: 0.2
    omega: 0.0
    rate_hz: 10.0
    max_duration: 5.0
    enabled: false
```

Values must have the type the node declared (`v` is a float: write `0.0`, not `0`). Launch arguments override nothing here; to change a parameter from the command line, add `-p` to a `ros2 run` (notebook 11) or edit the file and rebuild.

### The launcher of the exercise

When you run `dts code workbench`, the container executes the recipe's launcher, a small shell script:

```bash
# attach this container to the robot's zenoh router as a client
if [ -z "${ZENOH_CONFIG_OVERRIDE:-}" ] && [ -n "${VEHICLE_NAME:-}" ]; then
    export ZENOH_CONFIG_OVERRIDE="mode=\"client\";connect/endpoints=[\"tcp/${VEHICLE_NAME}.local:7447\"]"
fi
dt-exec ros2 launch ros2_basics drive_node.launch.py veh:="${VEHICLE_NAME}" car_cmd_topic:="${CAR_CMD_TOPIC:-car_cmd_switch_node/cmd}"
```

The two things every ROS 2 process of this learning experience needs: a client connection to the router of the robot (notebook 04) and the launch of the node in the robot's namespace.

## End to end: build, deploy, drive

Time to run the whole workflow from your laptop, the way the exercise is meant to be delivered. You should have these terminals on your laptop:

1. the code editor (`dts code editor`), showing this notebook;
2. (virtual robot only) the Duckiematrix (`dts code start_matrix`);
3. a terminal for `dts code build` and `dts code workbench`.

and the robot's ROS 2 stack up (`dts stack up -R ROBOTNAME ros2/duckiebot` and `dts stack up -R ROBOTNAME ros2-core/default`).

### Build

In terminal 3, from the root of this repository:

```bash
dts code build -R ROBOTNAME
```

The first execution takes some time, as it builds a Docker image; later builds are fast. Inside the output, the important part is the `colcon` build of your packages:

```shell
Starting >>> lx_ros2_tools
Starting >>> ros2_basics
Finished <<< lx_ros2_tools [1.1s]
Finished <<< ros2_basics [1.3s]

Summary: 2 packages finished [1.9s]
```

If you had a syntax error in your code, the build fails and reports the error, exactly as the `colcon build` in the editor did.

### Run

```bash
dts code workbench -R ROBOTNAME [-m]
```

with `-m` when the robot is virtual and the Duckiematrix is running. The command synchronises the code with your robot (it may ask for the `ssh` password, `quackquack` by default) and starts the launcher. The terminal shows the log of `drive_node`:

```shell
[INFO] [ROBOTNAME.drive_node]: ready: publishing on '/ROBOTNAME/car_cmd_switch_node/cmd' once started
```

### Drive, and watch every link of the chain

From the editor, run the cells below. They are the panels of notebook 09 plus a view of the whole data path of Figure 1 of notebook 02: the commands your node publishes, the wheel commands the kinematics node derives from them, the encoder ticks that prove the wheels turned, and the camera.

In [ ]:
ROBOT = "ROBOTNAME"   # <-- the name of your (real or virtual) Duckiebot

In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ParamPanel, ActionButtons, ImageView, TopicPlot

ros = RosSession(ROBOT)
panel = ParamPanel(ros, "drive_node", {"v": (-0.5, 0.5, 0.01), "omega": (-4.0, 4.0, 0.1), "max_duration": (0.0, 30.0, 0.5)})
buttons = ActionButtons(ros, "drive_node", ["start", "stop"])

In [ ]:
cmd_plot = TopicPlot(ros, "car_cmd_switch_node/cmd", "duckietown_msgs/msg/Twist2DStamped", ["v", "omega"],
                     title="car commands published by drive_node")
wheels_plot = TopicPlot(ros, "wheels_driver_node/wheels_cmd", "duckietown_msgs/msg/WheelsCmdStamped", ["vel_left", "vel_right"],
                        title="wheel commands produced by kinematics_node")
ticks_plot = TopicPlot(ros, "left_wheel_encoder_driver_node/tick", "duckietown_msgs/msg/WheelEncoderStamped", ["data"],
                       sensor=True, title="left wheel encoder ticks")
camera = ImageView(ros, "camera_node/image/compressed")

Press *start*. Within five seconds you should see, in order: `v` jump to `0.2` in the first plot, `vel_left` and `vel_right` rise in the second, the tick count climb in the third, and the view from the camera move. Each plot is one link of the chain; if one stays flat, the previous notebooks tell you how to investigate that link (`ros2 topic hz`, `ros2 node info`, the README's troubleshooting section).

Try an `omega` of `2.0` (press *Send* then *start*): the robot turns left, and the right wheel command is larger than the left one. Try the wiggle you wrote in the exercise of notebook 09.

The terminal equivalent of these plots, from an editor terminal pointed at the robot (notebook 08):

```bash
ros2 topic echo /ROBOTNAME/wheels_driver_node/wheels_cmd
```

In [ ]:
cmd_plot.close(); wheels_plot.close(); ticks_plot.close(); camera.close()

## Congratulations!

You have written a ROS 2 node that drives your Duckiebot (in simulation or on real hardware), and you know how to find out what any ROS 2 graph is made of:

- `ros2 node list / info`, `ros2 topic list / info / echo / hz`, `ros2 interface show`, `ros2 param`, `ros2 service` to inspect it (notebooks 03, 07, 08);
- the zenoh router and `VEHICLE_NAME` to reach the robot (notebook 04);
- `colcon build` and `source install/setup.bash` to make your package visible (notebooks 05, 06);
- `rclpy` nodes with timers, subscriptions, parameters and services (notebooks 09, 10);
- namespaces, remapping and launch files to deploy the same code anywhere (notebooks 11, 12).

Some ideas to go further with the same package:

- make `drive_node` drive a square (four straight segments and four turns, using `elapsed`);
- light the LEDs when the robot drives: publish a `duckietown_msgs/msg/LEDPattern` on `led_driver_node/led_pattern` from `srv_start` and `stop` (look at the message definition for the `rgb_vals` field);
- make the robot stop in front of an obstacle: subscribe to the time-of-flight sensor of `tof_driver_node` (find its topic and type with `ros2 node info`) and set `v` to zero when the distance is small;
- replace the constant command by something computed from the camera image: that is what the next learning experiences do.